# Two-stage Transfer Learning Task
In assignment 7A, a ChemBERTa model was fine-tuned to the ESOL dataset. Since that task took some considerable training time, the model was saved for further reuse, e.g. where only the regression head is retrained (which in contrast is a much cheaper operation).

Conceptually, this is a two-stage TL approach:

Foundation model ChemBERTa (general chemistry language) -> ESOL-tuned ChemBERTa (biased towards physicochemical descriptors) -> quick predictors (linear probes)

Reusing fine-tuned checkpoints as new model backbones is a routine operation to save computational time.

### Tasks
Note: The same random-state for splitting the dataset was used for all involved notebooks (`foundation_models.ipynb`, `7A_FineTuning.ipynb`).

1) Load the ESOL-tuned ChemBERTa model (encoder plus small regressor NN) and evaluate the predictions for the ESOL data (snippet provided)
2) In analogy to the notebook `foundation_models.ipynb` (session15/16), use the ESOL-tuned ChemBERTa model as fixed encoder and build a small machine learning model of your choice on top (e.g. ridge regression, RF, GB, ...)
3) Replace the dataset by the toxicity dataset (``tdc_ld50_zhu.csv``) and rerun the evaluation for the different transfer learning combinations (ChemBERTa+Regressor(retrain), ESOL-tuned ChemBERTa+Regressor(do not retrain), ESOL-tuned ChemBERTa+MLModel(retrain)), i.e. simply rerun the notebook with another dataset. Hint: you can crop the dataset size a bit by sampling so that retraining doesn't take too long (e.g. a GB model from task 2 took about 6 mins on my PC).
4) Complete the discussion points.


### Task 0: Import dependencies and data

In [55]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

Load the data including train test-split (use the same as in the other examples!)

In [56]:
#df = pd.read_csv("esol.csv")
df = pd.read_csv("tdc_ld50_zhu.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)

print(f"Dataset size: {len(df)}")
df.head()

Dataset size: 7376


,smiles,ld_50
0,[O-][N+](=Nc1ccccc1)c1ccccc1,2.505
1,BrC(Br)Br,2.343
2,C=CBr,2.330
3,Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465
4,S=C=Nc1ccc(Br)cc1,2.729


In [57]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [58]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        #label = self.df.iloc[idx]["logS"]
        label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


### Task 1:
Load and evaluate the ESOL-tuned ChemBERTa model.

In [59]:
# recreate model class
class chemberta_esol_regressor(nn.Module):
    
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc1 = nn.Linear(encoder.config.hidden_size, 256)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        x = self.act(self.fc1(cls))
        return self.fc2(x).squeeze(-1)

In [60]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>

Initialise the dataset and the loader.

In [61]:

test_dataset = ESOLDataset(val_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)


Evaluate the pretrained model:

In [62]:
# important: put model into evaluation mode (diables dropout and gradient)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        y_true.append(batch["labels"].numpy())
        y_pred.append(preds.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 5.776
Test MAE:  5.407
Test R²:   -33.627


### Task 2:
Use the ESOL-tuned ChemBERTa model as fixed encoder only and use its output as training input for a small ML model (not a NN) of your choice (= new trainable head). 

You define the encoder and tokenizer in analogy to the `foundation-models.ipynb`, likewise the smiles_encoding function, but you may have to change some small details.

Hint: Since the regression model is not a NN, you could use `return np.vstack(all_embeddings)` so that the embeddings are nicely compatible with any scikit-learn models.

define encoder and tokeniser + freeze layers

In [63]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [64]:

# no weights of the pretrained model are updated, used as fixed backbone
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False


smiles embedding

In [65]:
@torch.no_grad()
def embed_smiles(smiles_list, batch_size=32):
    all_embeddings = []

    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        outputs = encoder(**inputs)
        hidden = outputs.last_hidden_state           # (B, L, D)
        mask = inputs["attention_mask"].unsqueeze(-1)

        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
        all_embeddings.append(pooled)

    return np.vstack(all_embeddings)

smiles encoding

In [66]:
X_train = embed_smiles(train_df["smiles"].tolist())
#y_train = torch.tensor(train_df["logS"].values).float()
y_train = torch.tensor(train_df["ld_50"].values).float()

X_val = embed_smiles(val_df["smiles"].tolist())
#y_val = torch.tensor(val_df["logS"].values).float()
y_val = torch.tensor(val_df["ld_50"].values).float()

print("Train embeddings:", X_train.shape)
print("Validation embeddings:", X_val.shape)

Train embeddings: (5900, 768)
Validation embeddings: (1476, 768)


My ML Model ridge regression

In [67]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_basic = ridge.predict(X_val)

In [68]:
rmse = np.sqrt(mean_squared_error(y_val, pred_basic))
mae = mean_absolute_error(y_val, pred_basic)
r2 = r2_score(y_val, pred_basic)
print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 0.759
Test MAE:  0.586
Test R²:   0.401


save metrics ESOL

In [ ]:
esol_metrics = {
    "ChemBERTa+Regressor": "Test RMSE: 1.579, Test MAE:  1.241, Test R²:   0.251", 
    "ESOL-tuned ChemBERTa+Regressor": "Test RMSE: 1.025, Test MAE:  0.764, Test R²:   0.778",
    "ESOL-tuned ChemBERTa+MLModel": "Test RMSE: 1.021, Test MAE:  0.785, Test R²:   0.779"

}

ld_50_metrics={
    "ChemBERTa+Regressor" : "Test RMSE: 1.169, Test MAE:  0.887, Test R²:   0.021", 
    "ESOL-tuned ChemBERTa+Regressor" : "Test RMSE: 5.776, Test MAE:  5.407, Test R²:  -33.627",
    "ESOL-tuned ChemBERTa+MLModel" : "Test RMSE: 0.759, Test MAE:  0.586, Test R²:   0.401"
    }

### Task 3: Rerun with the toxicity dataset 
You can simply replace the imported dataframe. Note that depending on the model you chose in Task 2, training may take a bit - you can alleviate that problem by using a sample of the dataset.

You can run the General ChemBERTa + Regressor model in the original ``foundation_model.ipynb`` notebook (session 15/16).

### Task 4: Discussion

1) Why is it important for comparing the generalisation/performance of the different models to have the same random-state for the train-test split considering the fine-tuning in 7A and the evaluation in 7B? What would the predictions tell you otherwise?
2) How did the performances of the three approaches compare for the ESOL dataset? Did the transfer learning stages improve the models?
3) Smaller models trained on molecular descriptors based on the smiles strings in the ESOL dataset (e.g. a GB model), delivered:
- Train RMSE: 0.386
- Test RMSE: 0.776
- Train R2: 0.965
- Test R2: 0.873
How do you judge that in comparison to the much more complicated models?
4) Discuss the results for using the three approaches on the toxicity dataset. Which one performed best? What is a clear no-go? Comment on target vs. source tasks in this context.
5) What would be a better approach for the toxicity?
6) How could we generally improve the performance?

1) Reproducibility, consistencay in model eval, the params are set for this split and they allow us to directly compare the output metrics. Otherwise th epredictions would tell us how well the models generalise new data. 

2) Yes it significantly improved the performance (best). The worst was just the ChemBERTA+Regressor, the best was ESOL-tuned+Regressor. 
3) Higher performance than the more complicated models. Seems to be better suited, maybe this is a bad application for transfer learning.

4) The worst was ESOL-tuned ChemBERTa+Regressor by far (no-go) and tghe best was still ESOL-tuned+ML. I think the ML one performed best bc it allowed more adaptation to the dataset compared to a normal regressor model. Accordning to the numbers the pretraining helped, so I gather that the source and target tasks are similar (transfer learmning okay).

5) I guess fine tuning with a diff dataset than ESOL, maybe one more connected with metabolic stability. Or I saw online that people have had success with pairwise learning, did not read more into it tho (https://pubs.acs.org/doi/10.1021/acs.est.5c01289).

6) unfreeze more layers, maybe. 


> 3) we only have smiles for our foundational model, the small model has the mol descriptors (here ealsily ableitbar, but imagine not), then the foundational model would have a huge advantage

> 4. performed okay, underlying assumption: toxcicity is rel to structure

> 6. tox dataset, better data, more complex data, descriptors that we calced for other models


> @discussion point 3: Korrekt, die kleinen Modelle funktionieren oft recht gut, wenn man entsprechende Datensätze hat (in diesem Fall molekulare Deskriptoren). Allerdings zeigt der Parallelversuch mit dem Transfer Learning, dass man auch erstaunlich gut hinkommt, wenn man diese speziellen Daten nicht hat (das foundation model hatte ja nichts als die SMILES). 

> @discussion points 4&5: Genau, mal beiseite gelassen, was das R2 sagt, sind die Resultate eigentlich gar nicht so schlecht, wenn man einen generellen Encoder nutzt und dann einen Prediction head trainiert ("linear probe" - als transfer-learning-Ansatz). Könnte man natürlich noch verbessern, indem man das foundation model mit tox-Daten trainiert. Dabei schränkt sich natürlich das Feintuning nicht auf den Datensatz ein, den man zur Verfügung hat, sondern könnte alle möglichen relevanten Daten dazu verwenden. Joint learning oder ähnliche Ansätze können da natürlich auch Verbesserungen bringen - wie auch in dem Paper, das du gefunden hast (wenn ich das jetzt auf die Schnelle richtig rausgefiltert habe) - speziell wenn es um komplexere Systeme geht, wie etwa ein grösseres Ökosystem und nicht einfach nur einen einzelnen Parameter (e.g. LD50 für eine Maus), macht es klarerweise Sinn, die Zusammenhänge auch miteinzubeziehen. 